# 多周期共振分析

**目标**: 分析短中长期趋势的共振状态，判断市场14种阶段

**周期划分**:
- 短期: 1-8周 (5-40个交易日)
- 中期: 9-24周 (45-120个交易日)
- 长期: 25-48周 (125-240个交易日)

**共振类型**: 根据三个周期的趋势方向组合，判断14种市场阶段

In [ ]:
# 统一环境初始化（自动检测项目路径）from notebooks.lib import (    setup_research_environment,    ErrorBoundary,    ResultSaver)# 初始化研究环境env = setup_research_environment(verbose=True)# 导入必要的库import pandas as pdimport numpy as npfrom datetime import datetime, timedelta# 从环境获取组件jq = Nonetry:    jq = env.get_jqdata_client()except Exception as e:    print(f"⚠️ JQData初始化失败: {{e}}")# 加载配置config = env.load_config('config')INDEX_CODE = config.get('data', {{}}).get('default_index', '000001.XSHG')# 初始化结果保存器result_saver = ResultSaver("multi_period_resonance")print('✅ 环境加载完成')

## 1. 多周期趋势分析

In [ ]:
index_code = "000001.XSHG"
result = trend_analyzer.analyze_market(index_code=index_code)

if result:
    print("=" * 60)
    print("多周期趋势分析结果")
    print("=" * 60)
    
    print(f"\n📊 短期趋势 (1-8周):")
    print(f"  方向: {result.short_term.direction.value}")
    print(f"  得分: {result.short_term.score:.2f}")
    print(f"  置信度: {result.short_term.confidence:.2%}")
    print(f"  仓位建议: {result.short_term.position_suggestion:.0%}")
    
    print(f"\n📊 中期趋势 (9-24周):")
    print(f"  方向: {result.medium_term.direction.value}")
    print(f"  得分: {result.medium_term.score:.2f}")
    print(f"  置信度: {result.medium_term.confidence:.2%}")
    print(f"  仓位建议: {result.medium_term.position_suggestion:.0%}")
    
    print(f"\n📊 长期趋势 (25-48周):")
    print(f"  方向: {result.long_term.direction.value}")
    print(f"  得分: {result.long_term.score:.2f}")
    print(f"  置信度: {result.long_term.confidence:.2%}")
    print(f"  仓位建议: {result.long_term.position_suggestion:.0%}")
    
    print(f"\n🎯 综合评估:")
    print(f"  综合得分: {result.composite_score:.2f}")
    print(f"  市场阶段: {result.market_phase}")
    print(f"  整体方向: {result.overall_direction.value}")

## 2. 共振分析

In [ ]:
if result and result.resonance:
    print("=" * 60)
    print("多周期共振分析")
    print("=" * 60)
    
    resonance = result.resonance
    print(f"共振类型: {resonance.get('type', 'N/A')}")
    print(f"共振强度: {resonance.get('strength', 0):.2f}")
    print(f"共振描述: {resonance.get('description', 'N/A')}")
else:
    print("无共振数据")

In [ ]:
# 14种市场阶段说明
market_phases = {
    "强势牛市": "短+中+长全部上涨，共振最强",
    "牛市加速": "短期强于中长期，加速上涨",
    "牛市末期": "长期上涨但短期走弱，注意风险",
    "震荡偏强": "中长期上涨，短期震荡",
    "震荡市": "各周期方向不一致",
    "震荡偏弱": "中长期下跌，短期反弹",
    "熊市初期": "短期开始下跌，中长期还在高位",
    "熊市加速": "短+中+长全部下跌，风险最大",
    "熊市末期": "长期下跌但短期企稳",
    "底部探底": "短期反弹，中长期仍弱",
    "底部确认": "短中期开始上涨，长期企稳",
    "复苏初期": "短中期上涨，长期跟进",
    "突破在即": "各周期逐渐向上对齐",
    "顶部预警": "各周期开始出现分化"
}

print("\n市场阶段参考表:")
for phase, desc in market_phases.items():
    marker = "👉" if result and result.market_phase == phase else "  "
    print(f"{marker} {phase}: {desc}")

## 3. 可视化

In [ ]:
if result:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 三周期得分对比
    ax1 = axes[0]
    periods = ['短期', '中期', '长期']
    scores = [result.short_term.score, result.medium_term.score, result.long_term.score]
    colors = ['green' if s > 0 else 'red' for s in scores]
    
    bars = ax1.bar(periods, scores, color=colors, alpha=0.7)
    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax1.set_ylim(-100, 100)
    ax1.set_title('三周期趋势得分')
    ax1.set_ylabel('得分')
    
    for bar, score in zip(bars, scores):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.1f}', ha='center', va='bottom' if height > 0 else 'top')
    
    # 仓位建议
    ax2 = axes[1]
    positions = [result.short_term.position_suggestion, 
                 result.medium_term.position_suggestion,
                 result.long_term.position_suggestion]
    
    ax2.bar(periods, positions, color='blue', alpha=0.7)
    ax2.set_ylim(0, 1)
    ax2.set_title('各周期仓位建议')
    ax2.set_ylabel('仓位比例')
    
    for i, pos in enumerate(positions):
        ax2.text(i, pos + 0.02, f'{pos:.0%}', ha='center')
    
    plt.tight_layout()
    plt.show()

## 4. 多指数共振对比

In [ ]:
indices = {
    '000001.XSHG': '上证指数',
    '399001.XSHE': '深证成指',
    '399006.XSHE': '创业板指',
    '000300.XSHG': '沪深300'
}

print("=" * 80)
print("多指数共振分析")
print("=" * 80)
print(f"{'指数':<12} {'短期':<10} {'中期':<10} {'长期':<10} {'综合得分':<10} {'阶段':<15}")
print("-" * 80)

for code, name in indices.items():
    try:
        r = trend_analyzer.analyze_market(index_code=code)
        if r:
            short = f"{r.short_term.score:.1f}"
            medium = f"{r.medium_term.score:.1f}"
            long = f"{r.long_term.score:.1f}"
            composite = f"{r.composite_score:.1f}"
            phase = r.market_phase
            print(f"{name:<12} {short:<10} {medium:<10} {long:<10} {composite:<10} {phase:<15}")
    except Exception as e:
        print(f"{name:<12} 分析失败")

## 5. 保存研究结论

In [ ]:
if result:
    conclusion = {
        "index_code": index_code,
        "short_term": {"score": result.short_term.score, "direction": result.short_term.direction.value},
        "medium_term": {"score": result.medium_term.score, "direction": result.medium_term.direction.value},
        "long_term": {"score": result.long_term.score, "direction": result.long_term.direction.value},
        "composite_score": result.composite_score,
        "market_phase": result.market_phase,
        "resonance": result.resonance
    }
    
    save_research_conclusion(
        module="multi_period_resonance",
        findings=conclusion,
        recommendation=f"市场阶段: {result.market_phase}, 综合得分: {result.composite_score:.1f}",
        metadata={"index_code": index_code}
    )
    
    print("✅ 研究结论已保存")